# Data Profiling: HVFHV - NYC TLC

**Tipo de vehiculo:** High-Volume For-Hire Vehicle (Uber, Lyft - operadores masivos)

Este notebook realiza un perfil exploratorio completo del dataset HVFHV del NYC Taxi and Limousine Commission.
Se analizan estadisticas descriptivas, distribuciones temporales, zonas de pickup, patrones operativos,
tiempo de espera y viajes compartidos.

**Columnas disponibles:** `hvfhs_license_num`, `dispatching_base_num`, `originating_base_num`,
`request_datetime`, `on_scene_datetime`, `pickup_datetime`, `dropoff_datetime`, `PULocationID`,
`DOLocationID`, `trip_miles`, `trip_time`, `base_passenger_fare`, `tolls`, `bcf`, `sales_tax`,
`congestion_surcharge`, `airport_fee`, `tips`, `driver_pay`, `shared_request_flag`,
`shared_match_flag`, `access_a_ride_flag`, `wav_request_flag`, `wav_match_flag`

In [1]:
import os
os.makedirs('images', exist_ok=True)
# Configuracion del entorno y librerias
import sys
sys.path.insert(0, '/home/jovyan/work')

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import numpy as np

from pyspark.sql import functions as F
from src.spark_utils import get_spark, read_parquet
from src.paths import RAW

# Configuracion de estilo de graficos
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

VEHICLE = 'hvfhv'
RAW_PATH = str(RAW[VEHICLE])
print(f'Raw path: {RAW_PATH}')


Raw path: /home/jovyan/work/data/raw/hvfhv


## 1. Inicializacion de Spark

In [2]:
# Iniciar sesion de Spark y cargar datos crudos HVFHV
# Los archivos parquet tienen el prefijo fhvhv_*
spark = get_spark(app_name='HVFHV-Profiling')

# Leer todos los archivos parquet del directorio HVFHV
df = read_parquet(spark, RAW_PATH)

total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')
print()
print('Schema del dataset HVFHV:')
df.printSchema()


2026-07-18 19:33:21 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-18 19:33:26 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/hvfhv (Robust Mode)
Total de registros cargados: 715,550,152

Schema del dataset HVFHV:
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)

## 2. Perfil General: Estadisticas Descriptivas

In [3]:
# Estadisticas descriptivas generales
print('=== Estadisticas Descriptivas ===')
stats_pd = df.describe().toPandas().set_index('summary')
print(stats_pd.to_string())

print()
print('=== Conteo de Nulos por Columna ===')

# Calcular nulos y porcentaje por columna
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas()

null_df = null_counts.T.rename(columns={0: 'nulos'})
null_df['porcentaje'] = (null_df['nulos'] / total_registros * 100).round(2)
null_df = null_df.sort_values('porcentaje', ascending=False)
print(null_df.to_string())

=== Estadisticas Descriptivas ===
        hvfhs_license_num dispatching_base_num originating_base_num       PULocationID       DOLocationID          trip_miles           trip_time base_passenger_fare               tolls                 bcf          sales_tax congestion_surcharge          airport_fee                tips          driver_pay shared_request_flag shared_match_flag access_a_ride_flag wav_request_flag wav_match_flag  cbd_congestion_fee
summary                                                                                                                                                                                                                                                                                                                                                                                                                        
count           715550152            715550152            522708993          715550152          715550152           715550152         

## 3. Distribucion Temporal: Viajes por Ano y Mes

In [4]:
# Extraer anio-mes de pickup_datetime
df_temporal = df.withColumn('anio_mes', F.date_format('pickup_datetime', 'yyyy-MM'))

viajes_por_mes = df_temporal.groupBy('anio_mes') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy('anio_mes') \
    .toPandas()

viajes_por_mes = viajes_por_mes.dropna(subset=['anio_mes'])
viajes_por_mes = viajes_por_mes[
    viajes_por_mes['anio_mes'].str.match(r'^\d{4}-\d{2}$', na=False)
].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(
    range(len(viajes_por_mes)),
    viajes_por_mes['total_viajes'],
    color='darkorange',
    edgecolor='white',
    linewidth=0.5
)
ax.set_xticks(range(len(viajes_por_mes)))
ax.set_xticklabels(viajes_por_mes['anio_mes'], rotation=90, fontsize=7)
ax.set_xlabel('Periodo (Anio-Mes)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Distribucion Temporal de Viajes HVFHV por Anio y Mes')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
print(f'Periodos disponibles: {viajes_por_mes["anio_mes"].min()} a {viajes_por_mes["anio_mes"].max()}')

Periodos disponibles: 2023-01 a 2025-12


## 4. Top 10 Bases Operadoras (dispatching_base_num) y Licencias HVFHV

In [5]:
# Top 10 bases operadoras por volumen de viajes
top_bases = df.groupBy('dispatching_base_num') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy(F.desc('total_viajes')) \
    .limit(10) \
    .toPandas()

top_bases['dispatching_base_num'] = top_bases['dispatching_base_num'].fillna('(Sin Base)')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: top bases
colores = sns.color_palette('muted', n_colors=len(top_bases))
bases_rev = top_bases['dispatching_base_num'][::-1].values
viajes_rev = top_bases['total_viajes'][::-1].values
axes[0].barh(bases_rev, viajes_rev, color=colores)
axes[0].set_xlabel('Total de Viajes')
axes[0].set_title('Top 10 Bases Operadoras - HVFHV')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(viajes_rev):
    axes[0].text(v * 1.01, i, f'{v:,}', va='center', fontsize=8)

# Panel derecho: distribucion por licencia (hvfhs_license_num)
# HV0002=Juno, HV0003=Uber, HV0004=Via, HV0005=Lyft
mapa_licencias = {'HV0002': 'Juno', 'HV0003': 'Uber', 'HV0004': 'Via', 'HV0005': 'Lyft'}
licencias_dist = df.groupBy('hvfhs_license_num') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy(F.desc('total_viajes')) \
    .toPandas()
licencias_dist['operador'] = licencias_dist['hvfhs_license_num'].map(mapa_licencias).fillna('Otro')

colores_lic = sns.color_palette('Set2', n_colors=len(licencias_dist))
axes[1].bar(licencias_dist['operador'], licencias_dist['total_viajes'], color=colores_lic)
axes[1].set_xlabel('Operador (hvfhs_license_num)')
axes[1].set_ylabel('Total de Viajes')
axes[1].set_title('Distribucion por Operador HVFHV\n(HV0002=Juno, HV0003=Uber, HV0004=Via, HV0005=Lyft)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, (v, lbl) in enumerate(zip(licencias_dist['total_viajes'], licencias_dist['operador'])):
    axes[1].text(i, v * 1.01, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print('Distribucion por licencia (operador):')
print(licencias_dist[['hvfhs_license_num', 'operador', 'total_viajes']].to_string(index=False))

Distribucion por licencia (operador):
hvfhs_license_num operador  total_viajes
           HV0003     Uber     522242674
           HV0005     Lyft     193307478


## 5. Distribucion por Zonas (Pickup)

In [6]:
# Top 10 zonas de pickup por volumen de viajes
# Nota: columna PULocationID (mayuscula) para HVFHV
top_zonas = df.groupBy('PULocationID') \
    .agg(F.count('*').alias('total_viajes')) \
    .filter(F.col('PULocationID').isNotNull()) \
    .orderBy(F.desc('total_viajes')) \
    .limit(10) \
    .toPandas()

top_zonas['PULocationID'] = top_zonas['PULocationID'].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('Oranges_d', n_colors=len(top_zonas))
ax.bar(top_zonas['PULocationID'], top_zonas['total_viajes'], color=colores)
ax.set_xlabel('Zona de Pickup (PULocationID)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Top 10 Zonas de Pickup - HVFHV')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(top_zonas['total_viajes']):
    ax.text(i, v * 1.01, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print('Top 10 zonas de pickup:')
print(top_zonas.to_string(index=False))

Top 10 zonas de pickup:
PULocationID  total_viajes
         138      14036649
         132      12797980
          79       9374569
          61       9337710
         230       8753856
         161       8537965
         231       8126225
          68       7923975
          37       7788478
          76       7754242


## 6. Distribucion Horaria de Viajes

In [7]:
# Distribucion horaria: extraer hora de pickup_datetime
viajes_por_hora = df.withColumn('hora', F.hour('pickup_datetime')) \
    .filter(F.col('hora').isNotNull()) \
    .groupBy('hora') \
    .agg(F.count('*').alias('total_viajes')) \
    .orderBy('hora') \
    .toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    viajes_por_hora['hora'],
    viajes_por_hora['total_viajes'],
    marker='o', linewidth=2, color='darkorange', markersize=5
)
ax.fill_between(viajes_por_hora['hora'], viajes_por_hora['total_viajes'], alpha=0.2, color='darkorange')
ax.set_xlabel('Hora del Dia (0-23)')
ax.set_ylabel('Total de Viajes')
ax.set_title('Distribucion Horaria de Viajes HVFHV')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
hora_pico = int(viajes_por_hora.loc[viajes_por_hora['total_viajes'].idxmax(), 'hora'])
print(f'Hora pico de demanda: {hora_pico:02d}:00 hs')

Hora pico de demanda: 18:00 hs


## 7. Tiempo de Espera (Request a Pickup)

In [8]:
# â”€â”€ OPTIMIZADO: Histograma de tiempo de espera via Spark Bucketing â”€â”€
# En lugar de .sample(1%).toPandas() (millones de filas brutas), calculamos
# el histograma en Spark con bins de 30 segundos y solo descargamos ~60 filas.

from pyspark.ml.feature import Bucketizer
import numpy as _np

df_espera = df.filter(
    F.col('request_datetime').isNotNull() & F.col('pickup_datetime').isNotNull()
).withColumn(
    'wait_seconds',
    (F.unix_timestamp('pickup_datetime') - F.unix_timestamp('request_datetime')).cast('double')
).filter(
    (F.col('wait_seconds') >= 0) & (F.col('wait_seconds') <= 1800)
)

# Estadisticas de tiempo de espera (una sola pasada Spark)
stats_espera = df_espera.select(
    F.min('wait_seconds').alias('min_seg'),
    F.avg('wait_seconds').alias('avg_seg'),
    F.percentile_approx('wait_seconds', 0.5).alias('mediana_seg'),
    F.max('wait_seconds').alias('max_seg')
).toPandas()

print('Estadisticas de tiempo de espera (segundos, rango 0-1800):')
print(stats_espera.to_string(index=False))

# Histograma via Bucketizer: bins de 30 segundos â†’ 60 filas a Pandas
N_BINS = 60
splits = list(_np.linspace(0, 1800, N_BINS + 1).tolist())
splits[-1] = float('inf')
bkt = Bucketizer(splits=splits, inputCol='wait_seconds', outputCol='_bucket')
hist_pd = (
    bkt.transform(df_espera.select('wait_seconds'))
       .groupBy('_bucket')
       .agg(F.count('*').alias('freq'))
       .orderBy('_bucket')
       .toPandas()
)
step = 1800 / N_BINS
hist_pd['centro'] = (hist_pd['_bucket'] + 0.5) * step
mediana = float(stats_espera['mediana_seg'].values[0])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hist_pd['centro'], hist_pd['freq'], width=step * 0.9,
       color='darkorange', edgecolor='white', alpha=0.8)
ax.set_xlabel('Tiempo de Espera (segundos)')
ax.set_ylabel('Frecuencia (poblacion completa)')
ax.set_title('Distribucion del Tiempo de Espera - HVFHV (0 a 1800 segundos)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.axvline(x=mediana, color='red', linestyle='--', linewidth=1.5,
           label=f'Mediana: {mediana:.0f} seg')
ax.legend()
plt.tight_layout()
plt.show()


Estadisticas de tiempo de espera (segundos, rango 0-1800):
 min_seg    avg_seg  mediana_seg  max_seg
     0.0 284.235394        242.0   1800.0


In [9]:
# Distribucion de shared rides: shared_request_flag y shared_match_flag
shared_req = df.groupBy('shared_request_flag') \
    .agg(F.count('*').alias('total')) \
    .toPandas()
shared_req['shared_request_flag'] = shared_req['shared_request_flag'].fillna('Sin Registro')

shared_match = df.groupBy('shared_match_flag') \
    .agg(F.count('*').alias('total')) \
    .toPandas()
shared_match['shared_match_flag'] = shared_match['shared_match_flag'].fillna('Sin Registro')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
colores_pie = sns.color_palette('pastel', n_colors=max(len(shared_req), len(shared_match)))

# Pie: shared_request_flag
wedges1, texts1, auto1 = axes[0].pie(
    shared_req['total'],
    labels=shared_req['shared_request_flag'],
    autopct='%1.2f%%',
    colors=colores_pie[:len(shared_req)],
    startangle=90,
    pctdistance=0.80
)
for t in auto1:
    t.set_fontsize(9)
axes[0].set_title('shared_request_flag\n(Viaje compartido solicitado)')

# Pie: shared_match_flag
wedges2, texts2, auto2 = axes[1].pie(
    shared_match['total'],
    labels=shared_match['shared_match_flag'],
    autopct='%1.2f%%',
    colors=colores_pie[:len(shared_match)],
    startangle=90,
    pctdistance=0.80
)
for t in auto2:
    t.set_fontsize(9)
axes[1].set_title('shared_match_flag\n(Viaje compartido efectivamente emparejado)')

fig.suptitle('Distribucion de Viajes Compartidos - HVFHV', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('shared_request_flag:')
shared_req['pct'] = (shared_req['total'] / shared_req['total'].sum() * 100).round(2)
print(shared_req.to_string(index=False))
print()
print('shared_match_flag:')
shared_match['pct'] = (shared_match['total'] / shared_match['total'].sum() * 100).round(2)
print(shared_match.to_string(index=False))

shared_request_flag:
shared_request_flag     total   pct
                  N 693730985 96.95
                  Y  21819167  3.05

shared_match_flag:
shared_match_flag     total   pct
                N 705806268 98.64
                Y   9743884  1.36


In [10]:
# Finalizar sesion de Spark
spark.stop()
print('Profiling completado.')

Profiling completado.
